A entregar:
- Algoritmo escrito en lenguaje natural
- codigo python
- resultado del codigo python

NIVEL 1:

EJERCICIO 1: SELECCIÓN DE PERSONAL

In [55]:
inputs_test = [ 
    "Laura Serra", "608112233", 6, 3, 3, 3, 
    "Clara Gómez", "655221199", 3, 3, 3, 2, 
    "Sergio López", "644112211", 3, 2, 2, 2, 
    "Candidat Error", "611000000", 5, 7, 3, 2, # Català = 7 → NO VÀLID 
    "Marta Ruiz", "699887766", 2, 3, 2, 1, 
    "Marc Vidal", "612443322", 6, 3, 3, 3, 
    "Júlia Romero", "677998877", 5, 3, 3, 2, 
    "Pol Navarro", "699554433", 4, 3, 2, 2, 
    "David Ferrer", "677665544", 2, 1, 1, 1, 
    "Anna Paredes", "633221144", 1, 2, 1, 0,  
    "Anibal Paredes", "644272727", 1, 2, 1, 0,  
    "Núria Casals", "600112200", 8, 2, 3, 3 # 11è candidat     
] 

Lenguaje natural:
Por cada candidato se obtiene el nombre completo, número de contacto, años de experiencia, conocimientos de catalana (0-3), dominio de excel (0-3) y titulo academico (0-3). Según la puntuación tendrá una categoria asignada: >= 12 prioridad alta; 12 >= 8 candiadto adecuado; 8 > no prioritario.

A partir de los datos aportados se obtienen 6 datos por cada candidato. Teniendo en cuenta que es una estructura fija se crea un diccionario a partir de los nombres con un subdiccionario por cada candidato. 

Para mantener el orden de llegada (un diccionario no es indexado) se añade la variable orden. De esa forma, al llegar a 10 candidatos rompe (condición + break)

In [56]:

def obtenir_dades(formularios):
    info_candidatos = {}
    orden = 0
    for i in range(0, len(formularios),6):
        orden += 1
        if orden > 10:
            break
        nombre = formularios[i]
        info_candidatos[nombre] =  {
        "contacto" : formularios[i+1],
        "experiencia" : formularios[i+2],
        "nv_catalan" : formularios[i+3],
        "nv_excel" : formularios[i+4],
        "nv_academico" : formularios[i+5],
        "orden_llegada" : orden}

    return info_candidatos

Para validar los datos, se comprueba que cumplan con los rangos dados para las categorias nv_catalan, nv_excel y nv_academico. Cualquier valor fuera del rango 0-3 estará descalificado. Para registrar los usuarios que pueden seguir el proceso de selección (y los que no) se crea un diccionario con las llaves "apto" y "no apto" con una lista con los nombres de los candidatos. 

Una vez se sabe quien no es apto, se le borra de la candidatura.

In [57]:

def validad_dades(formularios):
    clean_candidatos = {"aptos":[],"no_aptos":[]}
    for nombre, categoria in formularios.items():
        if categoria["nv_catalan"] > 3 or categoria["nv_excel"] > 3 or categoria["nv_academico"] > 3:
            clean_candidatos["no_aptos"].append(nombre)
        else:
            clean_candidatos["aptos"].append(nombre)
    for nombre in clean_candidatos["no_aptos"]:
        del formularios[nombre]
    return formularios

Una vez filtrados los usuarios aptos, se calcula la puntuación teniendo en cuenta la experiencia y los puntos en las catgorias nv_catalan, nv_excel y nv_academico:

puntuación = años de experiencia * 1.5 + puntuación categorias

In [58]:
def calcular_puntuacion(formularios):
    for nombre in formularios.keys():
        p_experiencia = formularios[nombre]["experiencia"]*1.5
        p_categorias = formularios[nombre]["nv_catalan"]+formularios[nombre]["nv_excel"]+formularios[nombre]["nv_academico"]
        formularios[nombre]["puntuacion"] = p_experiencia + p_categorias
    return formularios

Con la puntuación se crean una serie de condiciones:

- si supera o iguala los 12p es prioritat alta; 
- mayor o igual a 8 es candidat adequat; 
- sino es no prioritari. 

El orden de las condiciones va en orden decreciente en los puntos. En el caso que tenga 13p no saltará la primera condición. Si pusieramos la condicion "puntuacion >=8" la primera se quedaría ahí, porque cumple la condición, pero le corresponderia la prioridad alta.

In [59]:
def classificar_candidat(formularios):
    for nombre in formularios.keys():
        if formularios[nombre]["puntuacion"] >= 12:
            formularios[nombre]["prioridad"] = "Prioritat Alta"
        elif formularios[nombre]["puntuacion"] >= 8:
            formularios[nombre]["prioridad"] = "Candidat adequat"
        else:
            formularios[nombre]["prioridad"] = "No prioritari"
    return formularios

Mostrar los candidatos individualmente. Se imprime el nombre y demás información en diferentes lineas. Cada usuario se separa por "------------------" para que se aprecie mejor la información. También se ha añadido información complementaria para facilitar la lectura.

In [60]:
def mostrar_individual(formularios):
    for nombre, categorias in formularios.items():
        print(f'Candidato nº {categorias["orden_llegada"]}')
        print(f"Nombre: {nombre}")
        print(f"Telefono: {categorias["contacto"]}")
        print(f"Años de experiencia: {categorias["experiencia"]}")
        print(f"Nivel de catalán: {categorias["nv_catalan"]}")
        print(f"Nivel de excel: {categorias["nv_excel"]}")
        print(f"Nivel academico: {categorias["nv_academico"]}")
        print(f"Puntuación: {categorias["puntuacion"]}")
        print(f"Prioridad: {categorias["prioridad"]}")
        print("-" * 30)


Para mostrar el informe final se ha obtenido de los formularios el nombre y las categorias (nv_catalan, experiencia...).

Se añaden los elementos imprescindibles del informe a una lista para mejorar la ordenación (un diccionario no se puede ordenar tan facilmente).

Al ordenar, el campo "prioritat" es de tipo texto, por lo que se ordena alfabeticamente. En este caso el orden seria: Candidat adequat; No prioritari; Prioritat alta (o viceversa). Para ordenarlo de la manera que queremos, se substituye por un 2-1-3 respectivamente.

sort() ordena por el primer campo que encuentra en la lista. Al ser una lista de listas, coge del candidato X el valor de prioritat. En caso de empate, obtiene el segundo valor del candidato X -> puntuación.

En sort() se ha definido "reverse=True" para que se ordene la prioridad y la puntuación de mayor a menor. Si no se ponia saldria de menor a mayor.

In [63]:
def mostrar_informe(formularios):
    listado = []
    orden_prioritario = {"Prioritat Alta":3,
                         "Candidat adequat":2,
                         "No prioritari":1}
    for nombre, categorias in formularios.items():
        listado.append([
            orden_prioritario[categorias["prioridad"]],
            categorias["puntuacion"],
            nombre,
            categorias["contacto"]])
    listado.sort(reverse=True)
    orden_prioritario = {3:"Prioritat Alta",
                        2:"Candidat adequat",
                        1:"No prioritari"}
    print("INFORME FINAL:")
    for prioridad, puntuacion, nombre, contacto in listado:
        print(f"Nombre del candidato: {nombre}. Contacto: {contacto}. Prioridad: {orden_prioritario[prioridad]}. Puntuación final: {puntuacion}.")

Todas las funciones juntas. Cada función va modificando los datos y se los pasa a la siguiente, excepto mostrar_individual y mostrar_informe que obtienen los datos ya arreglados de la funcion classificar_candidat.

In [79]:
def main_seleccio(formularios_candidatos):
    datos = obtenir_dades(formularios_candidatos)
    datos_validados = validad_dades(datos)
    puntos_candidatos = calcular_puntuacion(datos_validados)
    candidatos_clasificados = classificar_candidat(puntos_candidatos)
    mostrar_individual(candidatos_clasificados)
    mostrar_informe(candidatos_clasificados)
    

main_seleccio(inputs_test)
    

Candidato nº 1
Nombre: Laura Serra
Telefono: 608112233
Años de experiencia: 6
Nivel de catalán: 3
Nivel de excel: 3
Nivel academico: 3
Puntuación: 18.0
Prioridad: Prioritat Alta
------------------------------
Candidato nº 2
Nombre: Clara Gómez
Telefono: 655221199
Años de experiencia: 3
Nivel de catalán: 3
Nivel de excel: 3
Nivel academico: 2
Puntuación: 12.5
Prioridad: Prioritat Alta
------------------------------
Candidato nº 3
Nombre: Sergio López
Telefono: 644112211
Años de experiencia: 3
Nivel de catalán: 2
Nivel de excel: 2
Nivel academico: 2
Puntuación: 10.5
Prioridad: Candidat adequat
------------------------------
Candidato nº 5
Nombre: Marta Ruiz
Telefono: 699887766
Años de experiencia: 2
Nivel de catalán: 3
Nivel de excel: 2
Nivel academico: 1
Puntuación: 9.0
Prioridad: Candidat adequat
------------------------------
Candidato nº 6
Nombre: Marc Vidal
Telefono: 612443322
Años de experiencia: 6
Nivel de catalán: 3
Nivel de excel: 3
Nivel academico: 3
Puntuación: 18.0
Prioridad:

EJERCICIO 2: SEGMENTACIÓN POR RIESGO DE CESTA DE LA COMPRA

In [65]:
clients = [ 
    {"id_client": 1, "nom": "Laura Serra", "edat": 23, "grup_edat": "18-25", "codi_comportament": 4}, 
    {"id_client": 2, "nom": "Marc Vidal", "edat": 35, "grup_edat": "26-40", "codi_comportament": 3}, 
    {"id_client": 3, "nom": "Ana López", "edat": 29, "grup_edat": "26-40", "codi_comportament": 2}, 
    {"id_client": 4, "nom": "Joan Riera", "edat": 19, "grup_edat": "18-25", "codi_comportament": 2}, 
    {"id_client": 5, "nom": "Mónica Pérez", "edat": 47, "grup_edat": "41-60", "codi_comportament": 4}, 
    {"id_client": 6, "nom": "Luis García", "edat": 52, "grup_edat": "41-60", "codi_comportament": 1}, 
    {"id_client": 7, "nom": "Pilar Sánchez", "edat": 61, "grup_edat": "60+", "codi_comportament": 3}, 
    {"id_client": 8, "nom": "Eva Martín", "edat": 38, "grup_edat": "26-40", "codi_comportament": 0}, 
    {"id_client": 9, "nom": "Diego Romero", "edat": 24, "grup_edat": "18-25", "codi_comportament": 1}, 
    {"id_client": 10, "nom": "Núria Costa", "edat": 33, "grup_edat": "26-40", "codi_comportament": 4} 
] 

In [66]:
import pandas as pd
datos = pd.DataFrame(clients)
datos

,id_client,nom,edat,grup_edat,codi_comportament
0,1,Laura Serra,23,18-25,4
1,2,Marc Vidal,35,26-40,3
2,3,Ana López,29,26-40,2
3,4,Joan Riera,19,18-25,2
4,5,Mónica Pérez,47,41-60,4
5,6,Luis García,52,41-60,1
6,7,Pilar Sánchez,61,60+,3
7,8,Eva Martín,38,26-40,0
8,9,Diego Romero,24,18-25,1
9,10,Núria Costa,33,26-40,4


El diccionario de diccionarios (sin condiciones)

In [ ]:
def obtenir_segment_i_factor(data):
    data[codi_comportament]

In [ ]:
def obtenir_factor_edat(x):
    pass

In [ ]:
def calcular_risc(x):
    pass

Procesar_clients se alimenta de las funciones "obtenir_segment_i_factor", "obtenir_factor_edat" y "calcular_risc".

In [ ]:
def procesar_clients(x,y,z):
    pass

Resultado final: dataframe o diccionario final
Respuestas de las preguntas
Main marketing (todas las funciones)